# Online Shoppers Purchasing Intention — Decision Tree from Scratch (Class Weighting)

**Google Colab — complete reproducible experiment**

**Research question:** Does giving more weight to sessions with `Revenue = 1` improve the identification of purchases when the dataset is imbalanced?

We compare our decision tree implemented from scratch to a scikit-learn decision tree, with equal weights and several positive-class weights. Hyperparameters are chosen on **validation**, and the **test set** is evaluated only after selection.

**How to run:** Open this `.ipynb` in Google Colab, select **Runtime → Run all**, and upload `online_shoppers_intention.csv` when prompted. No `src` folder or VS Code project is needed.

## 1. Dataset and initial findings

The project dataset contains **12,330 sessions**: **10,422 non-purchases (84.53%)** and **1,908 purchases (15.47%)**. There are **125 duplicate rows** and no missing values in the original data.

Our hypothesis is that an unweighted decision tree may favor the much more frequent non-purchase class. We exclude `Revenue` (target) and `PageValues` (a conservative feature choice), keep 16 input features, and encode categorical variables into numerical columns.

**Dataset input:** use the CSV from your project. The upload is used directly; the notebook does not depend on a public URL.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from google.colab import files
from IPython.display import display

# Getting data from online_shoppers_intention
uploaded = files.upload()
csv_files = [name for name in uploaded if name.lower().endswith('.csv')]
if not csv_files:
    raise ValueError('Please upload online_shoppers_intention.csv')
data = pd.read_csv(csv_files[0])

print('Dataset shape:', data.shape)
print('Missing values:', int(data.isna().sum().sum()))
print('Duplicated rows:', int(data.duplicated().sum()))
print('Purchases and non-purchases:')
print(data['Revenue'].value_counts())
print('Purchase rate:', round(data['Revenue'].astype(int).mean() * 100, 2), '%')
assert 'Revenue' in data.columns

In [ ]:
# Bar chart of the target distribution
counts = data['Revenue'].astype(int).value_counts().sort_index()
plt.figure(figsize=(6, 4))
plt.bar(['No purchase (0)', 'Purchase (1)'], [counts[0], counts[1]])
plt.title('Target distribution')
plt.ylabel('Number of shopping sessions')
plt.show()

## 2. Split the dataset and prepare the features

Use the original project methodology: `np.random.default_rng(42).permutation`, **60% train / 20% validation / 20% test**. Unlike a stratified split, this split does not force identical purchase rates across subsets.

One-hot encoding is fitted to the **training features**; validation/test columns are aligned to the training columns. We retain the same 16 features as the original scripts.

In [ ]:
# Our input variables
features = [
    'Administrative', 'Administrative_Duration',
    'Informational', 'Informational_Duration',
    'ProductRelated', 'ProductRelated_Duration',
    'BounceRates', 'ExitRates', 'SpecialDay',
    'Month', 'OperatingSystems', 'Browser',
    'Region', 'TrafficType', 'VisitorType', 'Weekend'
]
categories = [
    'Month', 'OperatingSystems', 'Browser', 'Region',
    'TrafficType', 'VisitorType', 'Weekend'
]

X = data[features]
y = data['Revenue'].astype(int)

# Shuffle the row indices
n = len(data)
indices = np.random.default_rng(42).permutation(n)
n_test = round(0.2 * n)
n_val = round(0.2 * n)
test_idx = indices[:n_test]
val_idx = indices[n_test:n_test + n_val]
train_idx = indices[n_test + n_val:]

X_train = X.iloc[train_idx].copy()
X_val = X.iloc[val_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].to_numpy()
y_val = y.iloc[val_idx].to_numpy()
y_test = y.iloc[test_idx].to_numpy()

# Convert categories to text
for column in categories:
    X_train[column] = X_train[column].astype(str)
    X_val[column] = X_val[column].astype(str)
    X_test[column] = X_test[column].astype(str)

# Convert categories into numerical columns
train_encoded = pd.get_dummies(X_train, columns=categories, dtype=float)
val_encoded = pd.get_dummies(X_val, columns=categories, dtype=float)
test_encoded = pd.get_dummies(X_test, columns=categories, dtype=float)
val_encoded = val_encoded.reindex(columns=train_encoded.columns, fill_value=0.0)
test_encoded = test_encoded.reindex(columns=train_encoded.columns, fill_value=0.0)
X_train_ready = train_encoded.to_numpy(dtype=float)
X_val_ready = val_encoded.to_numpy(dtype=float)
X_test_ready = test_encoded.to_numpy(dtype=float)

print('Training shape:', X_train_ready.shape)
print('Validation shape:', X_val_ready.shape)
print('Test shape:', X_test_ready.shape)
print('Purchase rates:', *(f'{z.mean()*100:.2f}%' for z in (y_train, y_val, y_test)))
assert X_train_ready.shape[1] == X_val_ready.shape[1] == X_test_ready.shape[1]
assert np.isfinite(X_train_ready).all()

## 3. Implement Gini impurity and the decision tree from scratch

For a node containing labels \(y_i\) and observation weights \(w_i\), the **weighted purchase proportion** is

\[p_w = \frac{\sum_i w_i \mathbf{1}(y_i=1)}{\sum_i w_i}.\]

The weighted Gini impurity remains \(G = 1-p_w^2-(1-p_w)^2\), and child impurities are averaged using the **sum of observation weights**, not just the number of observations. At a leaf, prediction also uses the **weighted majority vote**.

`purchase_weight = 1.0` reproduces the ordinary equal-weight tree. Values above 1 penalize mistakes on purchases more strongly.

We retain the original project's threshold candidates (up to 15 quantile values when a feature has many unique values). This is an approximation and may differ from scikit-learn's exhaustive splitting.

In [ ]:
# Calculating Gini impurity
def gini(y, sample_weight=None):
    y = np.asarray(y)
    if len(y) == 0:
        return 0.0
    if sample_weight is None:
        sample_weight = np.ones(len(y))
    total_weight = np.sum(sample_weight)
    if total_weight == 0:
        return 0.0
    p = np.sum(sample_weight[y == 1]) / total_weight
    return 1 - p**2 - (1 - p)**2

# Calculating weighted Gini impurity
def weighted_gini(y_left, y_right, w_left=None, w_right=None):
    if w_left is None:
        w_left = np.ones(len(y_left))
    if w_right is None:
        w_right = np.ones(len(y_right))
    total_left = np.sum(w_left)
    total_right = np.sum(w_right)
    total = total_left + total_right
    if total == 0:
        return 0.0
    return (total_left / total) * gini(y_left, w_left) + (total_right / total) * gini(y_right, w_right)

# Finding the best feature and threshold
def best_split(X, y, min_samples_leaf=20, sample_weight=None):
    if sample_weight is None:
        sample_weight = np.ones(len(y))
    best_feature = None
    best_threshold = None
    best_score = gini(y, sample_weight)
    for j in range(X.shape[1]):
        values = np.unique(X[:, j])
        if len(values) <= 1:
            continue
        if len(values) <= 16:
            thresholds = (values[:-1] + values[1:]) / 2
        else:
            thresholds = np.unique(np.quantile(values, np.linspace(0, 1, 17)[1:-1]))
        for t in thresholds:
            left = X[:, j] <= t
            right = ~left
            if left.sum() < min_samples_leaf or right.sum() < min_samples_leaf:
                continue
            score = weighted_gini(y[left], y[right], sample_weight[left], sample_weight[right])
            if score < best_score - 1e-12:
                best_score = score
                best_feature = j
                best_threshold = t
    return best_feature, best_threshold

# Building the decision tree
def build_tree(X, y, depth=0, max_depth=5, min_samples_leaf=20, sample_weight=None):
    if sample_weight is None:
        sample_weight = np.ones(len(y))
    weight_0 = np.sum(sample_weight[y == 0])
    weight_1 = np.sum(sample_weight[y == 1])
    prediction = int(weight_1 >= weight_0)
    if depth >= max_depth or gini(y, sample_weight) <= 1e-12:
        return {'prediction': prediction}
    j, t = best_split(X, y, min_samples_leaf=min_samples_leaf, sample_weight=sample_weight)
    if j is None:
        return {'prediction': prediction}
    left = X[:, j] <= t
    right = ~left
    return {
        'feature': j, 'threshold': t,
        'left': build_tree(X[left], y[left], depth+1, max_depth, min_samples_leaf, sample_weight[left]),
        'right': build_tree(X[right], y[right], depth+1, max_depth, min_samples_leaf, sample_weight[right])
    }

# Predicting one shopping session
def predict_one(x, tree):
    if 'prediction' in tree:
        return tree['prediction']
    if x[tree['feature']] <= tree['threshold']:
        return predict_one(x, tree['left'])
    return predict_one(x, tree['right'])

# Predicting all shopping sessions
def predict(X, tree):
    return np.array([predict_one(x, tree) for x in X], dtype=int)

In [ ]:
# Checkpoints
assert np.isclose(gini(np.array([0, 0, 0, 0])), 0.0)
assert np.isclose(gini(np.array([0, 0, 1, 1])), 0.5)
assert np.isclose(gini(np.array([0, 0, 0, 0, 1])), 0.32)
assert np.isclose(weighted_gini(np.array([0, 0]), np.array([1, 1])), 0.0)
assert np.isclose(weighted_gini(np.array([0, 1]), np.array([0, 1])), 0.5)
assert np.isclose(weighted_gini(np.array([0, 0, 1]), np.array([1])), 1 / 3)
X_small = np.array([[1], [2], [3], [4]], dtype=float)
y_small = np.array([0, 0, 1, 1])
j, t = best_split(X_small, y_small, min_samples_leaf=1)
assert j == 0 and np.isclose(t, 2.5)
small_tree = build_tree(X_small, y_small, max_depth=2, min_samples_leaf=1)
assert np.array_equal(predict(X_small, small_tree), y_small)
print('OK - Gini, weighted Gini, best split, and tree predictions')

## 4. Validation: compare equal weights and higher purchase weights

We search **3 tree depths** (`3, 5, 7`) × **5 positive-class weights** (`1, 2, 3, 5, majority/minority ratio`) × **2 implementations** (our tree, scikit-learn) = **30 model configurations**.

The weight ratio is computed **only from training labels**. We choose configurations by validation F1-score, never by test performance. We separately select the best scratch tree with equal weights and the best scratch tree across all positive weights.

In [ ]:
# Calculating classification metrics
def classification_metrics(y, y_pred):
    tp = np.sum((y == 1) & (y_pred == 1))
    fp = np.sum((y == 0) & (y_pred == 1))
    fn = np.sum((y == 1) & (y_pred == 0))
    accuracy = np.mean(y == y_pred)
    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    return {'Accuracy': accuracy, 'Precision': precision, 'Recall': recall, 'F1': f1}

# Comparing tree with scikit-learn
results = []
purchase_ratio = np.sum(y_train == 0) / np.sum(y_train == 1)
purchase_weights = [1.0, 2.0, 3.0, 5.0, float(purchase_ratio)]

for depth in [3, 5, 7]:
    for purchase_weight in purchase_weights:
        sample_weight = np.where(y_train == 1, purchase_weight, 1.0)
        # Training decision tree
        tree = build_tree(X_train_ready, y_train, max_depth=depth, min_samples_leaf=20, sample_weight=sample_weight)
        # Evaluating decision tree
        y_pred = predict(X_val_ready, tree)
        results.append({'Model': 'Our tree', 'Depth': depth, 'Purchase_weight': purchase_weight,
                        **classification_metrics(y_val, y_pred)})
        # Training scikit-learn decision tree
        sklearn_tree = DecisionTreeClassifier(criterion='gini', max_depth=depth, min_samples_leaf=20, random_state=42)
        sklearn_tree.fit(X_train_ready, y_train, sample_weight=sample_weight)
        # Evaluating scikit-learn decision tree
        y_pred = sklearn_tree.predict(X_val_ready)
        results.append({'Model': 'Scikit-learn', 'Depth': depth, 'Purchase_weight': purchase_weight,
                        **classification_metrics(y_val, y_pred)})

# Displaying comparison results
table = pd.DataFrame(results)
display(table.round(4))

# Selecting best model
best = table.loc[table['F1'].idxmax()]
our_results = table[table['Model'] == 'Our tree']
best_our_tree = our_results.loc[our_results['F1'].idxmax()]
equal_results = our_results[np.isclose(our_results['Purchase_weight'], 1.0)]
best_equal = equal_results.loc[equal_results['F1'].idxmax()]
print('Best overall model:', best['Model'])
print('Best scratch tree depth:', int(best_our_tree['Depth']))
print('Best scratch tree purchase weight:', round(best_our_tree['Purchase_weight'], 4))
print('Best scratch tree validation F1:', round(best_our_tree['F1'], 4))
assert len(table) == 30

In [ ]:
# Comparing equal weights with weighted purchases on validation
validation_comparison = pd.DataFrame([
    {'Method': 'Equal weights', **best_equal.to_dict()},
    {'Method': 'Weighted purchases', **best_our_tree.to_dict()}
])
display(validation_comparison[['Method', 'Depth', 'Purchase_weight', 'Accuracy', 'Precision', 'Recall', 'F1']].round(4))

# Comparing with baseline (?) check again!
baseline = np.zeros(len(y_val), dtype=int)
baseline_scores = classification_metrics(y_val, baseline)
print('No-purchase baseline (validation):', {k: round(v, 4) for k, v in baseline_scores.items()})

plt.figure(figsize=(7, 4))
plt.bar(['Equal weights', 'Weighted purchases'], [best_equal['F1'], best_our_tree['F1']])
plt.title('Validation F1-score: scratch decision tree')
plt.ylabel('F1-score')
plt.ylim(0, 0.55)
plt.show()

## 5. Final test — evaluated only after model selection

We train **two selected scratch-tree configurations** on the training split and evaluate each on the **same untouched test split**. We also report the trivial baseline that **always predicts no purchase (0)**. This baseline is **not** the equal-weight tree.

Confusion-matrix layout below: rows = actual class (`0`, `1`), columns = predicted class (`0`, `1`).

In [ ]:
# Training the selected decision tree
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

def test_model(name, depth, purchase_weight):
    sample_weight = np.where(y_train == 1, purchase_weight, 1.0)
    tree = build_tree(X_train_ready, y_train, max_depth=int(depth), min_samples_leaf=20, sample_weight=sample_weight)
    # Predicting test data
    y_pred = predict(X_test_ready, tree)
    # Calculating test results
    scores = classification_metrics(y_test, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
    print()
    print('Final test results:', name)
    print('Depth:', int(depth))
    print('Purchase weight:', round(purchase_weight, 4))
    for metric, value in scores.items():
        print(metric + ':', round(value, 4))
    print('Confusion matrix: TN =', tn, 'FP =', fp, 'FN =', fn, 'TP =', tp)
    assert len(y_pred) == len(y_test)
    assert set(np.unique(y_pred)).issubset({0, 1})
    return {'Model': name, 'Depth': int(depth), 'Purchase_weight': purchase_weight, **scores}, y_pred

# Comparing equal weights with weighted purchases
equal_test, equal_pred = test_model('Equal weights', best_equal['Depth'], 1.0)
weighted_test, weighted_pred = test_model('Weighted purchases', best_our_tree['Depth'], best_our_tree['Purchase_weight'])

# Displaying comparison results
final_comparison = pd.DataFrame([equal_test, weighted_test])
print('Final test comparison:')
display(final_comparison.round(4))

# Comparing with baseline
baseline_test = np.zeros(len(y_test), dtype=int)
baseline_scores_test = classification_metrics(y_test, baseline_test)
print('Baseline test results:', {k: round(v, 4) for k, v in baseline_scores_test.items()})

# Displaying confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, title, pred in zip(axes, ['Equal weights', 'Weighted purchases'], [equal_pred, weighted_pred]):
    ConfusionMatrixDisplay(confusion_matrix(y_test, pred, labels=[0, 1]), display_labels=['No purchase', 'Purchase']).plot(ax=ax, colorbar=False)
    ax.set_title(title)
plt.tight_layout()
plt.show()
print('OK - final test completed')

## 6. What we realized — interpretation of the experiment

The following values are **results from our previous run with this dataset, seed and code**. Re-running the notebook should reproduce them if the uploaded CSV is the same.

| Test metric | Equal weights | Weighted purchases |
|---|---:|---:|
| Max depth | 7 | 5 |
| Purchase weight | 1.0000 | 5.5702 |
| Accuracy | **0.8382** | 0.7283 |
| Precision | **0.4000** | 0.3132 |
| Recall | 0.1055 | **0.6438** |
| F1-score | 0.1670 | **0.4214** |
| True positives | 40 | **244** |
| False negatives | 339 | **135** |
| False positives | **60** | 535 |

**1. The dataset is imbalanced.** About 84.5% of all sessions are non-purchases. A naive no-purchase baseline can obtain high accuracy while missing every actual purchase. On our test split, the baseline had **84.63% accuracy but 0 recall and 0 F1**.

**2. The unweighted tree was biased toward non-purchase predictions.** Of **379 real purchases** in the test set, the equal-weight tree identified just **40** (10.55% recall).

**3. Increasing purchase weights improved our objective.** The weighted tree identified **244 of 379 purchases** (64.38% recall), which is **204 additional detected purchases**. Test F1 improved from **0.1670 to 0.4214** (about **2.52 times**).

**4. This improvement has a cost.** False-positive predictions rose from **60 to 535**, and accuracy and precision fell. Weighting makes the classifier more sensitive to purchases but less selective. It is therefore more useful **if missing a purchase is more costly than a false alarm**; the best trade-off depends on the application.

**5. We did not just change the weight.** Validation also selected a different depth (**7 → 5**). So this experiment demonstrates an improvement from **model selection with class weighting**, not the isolated causal effect of class weights at a fixed depth. A fixed-depth comparison would investigate that question more directly.

**6. Why scikit-learn can differ.** Our scratch tree uses a limited set of threshold candidates for continuous features, while scikit-learn considers more possible split points. Equal hyperparameters do not guarantee identical trees.

**Conclusion:** Our hypothesis was supported by validation and test results: class weighting greatly improved purchase detection and the F1-score, but the model still produced many false positives. Reporting both recall and precision is essential; accuracy alone is misleading for this problem.

## 7. Limitations and possible next steps

- **Duplicate rows:** the original data has 125 duplicates. Random splitting can place duplicate sessions in different sets; investigate and document this before claiming full independence.
- **Robustness:** the experiment uses one random split (seed 42). Repeated or stratified splits/cross-validation would better assess result variability.
- **Selection:** the test set was not used to choose weights or depths. Keep it that way; don't tune parameters using the final test result.
- **Business objective:** a different cost of false negatives vs false positives could favor a different weight or probability threshold.
- **Features:** `PageValues` was intentionally excluded, consistent with the original project.

**Takeaway for the report:** *Class weighting successfully addressed the tendency of an unweighted decision tree to ignore the minority purchase class, increasing recall and F1 at the expense of precision and accuracy.*